# CRSS Data Loading

**Purpose:** Load the original CRSS tables, explore the 3 tables(accidents, vehicles and persons), document the selected fields and coding, and create a cleaned combined table.

In [1]:
from pathlib import Path
from zipfile import ZipFile

import pandas as pd

DATA_PATH = Path("data/CRSS2024_three_tables.zip")
print(DATA_PATH.resolve())

C:\Users\maina\AppData\Local\Temp\crss_eda_930c4a67647d4c818a81294cd64142d9\data\CRSS2024_three_tables.zip


## Load the three original tables

Read every column in the three source CSV files so the raw structure remains visible before selection or transformation.

In [2]:
def load_crss(zip_path):
    with ZipFile(zip_path) as archive:
        names = {Path(member).name.lower(): member for member in archive.namelist()}
        return {
            "accident": pd.read_csv(archive.open(names["accident.csv"]), encoding="utf-8-sig", low_memory=False),
            "vehicle": pd.read_csv(archive.open(names["vehicle.csv"]), encoding="utf-8-sig", low_memory=False),
            "person": pd.read_csv(archive.open(names["person.csv"]), encoding="utf-8-sig", low_memory=False),
        }

raw = load_crss(DATA_PATH)

pd.DataFrame({
    "table": ["accident.csv", "vehicle.csv", "person.csv"],
    "rows": [len(raw["accident"]), len(raw["vehicle"]), len(raw["person"])],
    "columns": [len(raw["accident"].columns), len(raw["vehicle"].columns), len(raw["person"].columns)],
})

,table,rows,columns
0,accident.csv,51658,80
1,vehicle.csv,90641,167
2,person.csv,126159,112


## Explore accident.csv

This section describes `accident.csv` before a crash key, target, or feature set is finalized.

In [3]:
accident = raw["accident"]

print("Shape:", accident.shape)
print("Columns:")
print(accident.columns.tolist())

Shape: (51658, 80)
Columns:
['CASENUM', 'PSU', 'PSU_VAR', 'PSUSTRAT', 'REGION', 'REGIONNAME', 'URBANICITY', 'URBANICITYNAME', 'STRATUM', 'STRATUMNAME', 'PJ', 'WEIGHT', 'PEDS', 'PERNOTMVIT', 'VE_TOTAL', 'VE_FORMS', 'PVH_INVL', 'PERMVIT', 'MONTH', 'MONTHNAME', 'DAY_WEEK', 'DAY_WEEKNAME', 'WKDY_IM', 'WKDY_IMNAME', 'YEAR', 'YEARNAME', 'HOUR', 'HOURNAME', 'HOUR_IM', 'HOUR_IMNAME', 'MINUTE', 'MINUTENAME', 'MINUTE_IM', 'MINUTE_IMNAME', 'HARM_EV', 'HARM_EVNAME', 'EVENT1_IM', 'EVENT1_IMNAME', 'MAN_COLL', 'MAN_COLLNAME', 'MANCOL_IM', 'MANCOL_IMNAME', 'RELJCT1', 'RELJCT1NAME', 'RELJCT1_IM', 'RELJCT1_IMNAME', 'RELJCT2', 'RELJCT2NAME', 'RELJCT2_IM', 'RELJCT2_IMNAME', 'TYP_INT', 'TYP_INTNAME', 'REL_ROAD', 'REL_ROADNAME', 'WRK_ZONE', 'WRK_ZONENAME', 'LGT_COND', 'LGT_CONDNAME', 'LGTCON_IM', 'LGTCON_IMNAME', 'WEATHER', 'WEATHERNAME', 'WEATHR_IM', 'WEATHR_IMNAME', 'SCH_BUS', 'SCH_BUSNAME', 'INT_HWY', 'INT_HWYNAME', 'MAX_SEV', 'MAX_SEVNAME', 'MAXSEV_IM', 'MAXSEV_IMNAME', 'NUM_INJ', 'NUM_INJNAME', 'NO_INJ

In [4]:
accident.head(10)

,CASENUM,PSU,PSU_VAR,PSUSTRAT,REGION,REGIONNAME,URBANICITY,URBANICITYNAME,STRATUM,STRATUMNAME,...,MAXSEV_IM,MAXSEV_IMNAME,NUM_INJ,NUM_INJNAME,NO_INJ_IM,NO_INJ_IMNAME,ALCOHOL,ALCOHOLNAME,ALCHL_IM,ALCHL_IMNAME
0,202405532075,33,33,11,3,"South (MD, DE, DC, WV, VA, KY, TN, NC, SC, GA,...",1,Urban Area,10,Stratum 10 - Other,...,0,No Apparent Injury (O),0,No Person Injured/Property Damage Only Crash,0,No Person Injured/Property Damage Only Crash,2,No Alcohol Involved,2,No Alcohol Involved
1,202405532153,22,22,5,1,"Northeast (PA, NJ, NY, NH, VT, RI, MA, ME, CT)",2,Rural Area,9,Stratum 9 - LMY PV No Injuries in Crash,...,0,No Apparent Injury (O),0,No Person Injured/Property Damage Only Crash,0,No Person Injured/Property Damage Only Crash,2,No Alcohol Involved,2,No Alcohol Involved
2,202405532158,39,39,7,2,"Midwest (OH, IN, IL, MI, WI, MN, ND, SD, NE, I...",1,Urban Area,10,Stratum 10 - Other,...,0,No Apparent Injury (O),0,No Person Injured/Property Damage Only Crash,0,No Person Injured/Property Damage Only Crash,2,No Alcohol Involved,2,No Alcohol Involved
3,202405533474,25,25,8,2,"Midwest (OH, IN, IL, MI, WI, MN, ND, SD, NE, I...",1,Urban Area,9,Stratum 9 - LMY PV No Injuries in Crash,...,0,No Apparent Injury (O),0,No Person Injured/Property Damage Only Crash,0,No Person Injured/Property Damage Only Crash,9,Reported as Unknown,2,No Alcohol Involved
4,202405533487,25,25,8,2,"Midwest (OH, IN, IL, MI, WI, MN, ND, SD, NE, I...",1,Urban Area,10,Stratum 10 - Other,...,0,No Apparent Injury (O),0,No Person Injured/Property Damage Only Crash,0,No Person Injured/Property Damage Only Crash,9,Reported as Unknown,2,No Alcohol Involved
5,202405533497,47,47,9,2,"Midwest (OH, IN, IL, MI, WI, MN, ND, SD, NE, I...",2,Rural Area,8,Stratum 8 - NLMY PV Minor Injury,...,2,Suspected Minor Injury (B),1,1,1,1,9,Reported as Unknown,2,No Alcohol Involved
6,202405533502,67,67,8,2,"Midwest (OH, IN, IL, MI, WI, MN, ND, SD, NE, I...",1,Urban Area,8,Stratum 8 - NLMY PV Minor Injury,...,1,Possible Injury (C),1,1,1,1,9,Reported as Unknown,2,No Alcohol Involved
7,202405533521,83,83,13,3,"South (MD, DE, DC, WV, VA, KY, TN, NC, SC, GA,...",1,Urban Area,6,Stratum 6 - LMY PV Minor Injury,...,1,Possible Injury (C),2,2,2,2,2,No Alcohol Involved,2,No Alcohol Involved
8,202405533542,83,83,13,3,"South (MD, DE, DC, WV, VA, KY, TN, NC, SC, GA,...",1,Urban Area,8,Stratum 8 - NLMY PV Minor Injury,...,2,Suspected Minor Injury (B),1,1,1,1,2,No Alcohol Involved,2,No Alcohol Involved
9,202405533564,83,83,13,3,"South (MD, DE, DC, WV, VA, KY, TN, NC, SC, GA,...",1,Urban Area,10,Stratum 10 - Other,...,0,No Apparent Injury (O),0,No Person Injured/Property Damage Only Crash,0,No Person Injured/Property Damage Only Crash,2,No Alcohol Involved,2,No Alcohol Involved


### Missing values

In [5]:
accident.isna().sum().sort_values(ascending=False).to_frame("missing_values")

,missing_values
WRK_ZONENAME,50731
CASENUM,0
PSU_VAR,0
PSU,0
REGION,0
...,...
NO_INJ_IMNAME,0
ALCOHOL,0
ALCOHOLNAME,0
ALCHL_IM,0


### Unique value counts

In [6]:
accident.nunique(dropna=False).sort_values().to_frame("unique_values")

,unique_values
YEAR,1
YEARNAME,1
URBANICITY,2
URBANICITYNAME,2
RELJCT1_IMNAME,2
...,...
MINUTENAME,61
PSU_VAR,67
PJ,270
WEIGHT,1805


## Selected-field and coding dictionary

The proposal uses 10 crash-context predictors, one outcome source, four survey fields, and one crash identifier.

| Group | Field(s) | Meaning | ML treatment |
|---|---|---|---|
| Identifier | `CASENUM` | Identifies one crash | Not a predictor |
| Outcome source | `MAX_SEV` | Identifies the maximum reported injury severity | Not a predictor |
| Survey fields | `WEIGHT`, `PSU`, `PSU_VAR`, `PSUSTRAT` | Sampling weight and survey-design information | Retained separately |
| Numeric context | `VE_FORMS`, `PVH_INVL` | Vehicle counts | Numeric predictors |
| Categorical context | `MONTH`, `DAY_WEEK`, `HOUR`, `LGT_COND`, `WEATHER`, `RELJCT2`, `TYP_INT`, `WRK_ZONE` | Timing, environment, junction, and work-zone information | One-hot encode later |
| Source validation | `YEAR` | Confirms the release year | Not a predictor |

### Field codes

| Field | Meaning | Codes or values |
|---|---|---|
| `MONTH` | Calendar month | `1` January through `12` December |
| `DAY_WEEK` | Day of week | `1` Sunday through `7` Saturday |
| `HOUR` | Hour | `0` through `23`; `99` unknown |
| `LGT_COND` | Lighting | `1` daylight, `2` dark/not lighted, `3` dark/lighted, `4` dawn, `5` dusk, `6` dark/unknown, `7` other, `8` not reported, `9` unknown |
| `WEATHER` | Weather | Coded condition; `98` not reported and `99` unknown |
| `RELJCT2` | Junction relationship | Coded junction location; `98` not reported and `99` unknown |
| `TYP_INT` | Intersection type | Coded intersection configuration; `98` not reported and `99` unknown |
| `WRK_ZONE` | Work-zone condition | `0` none, `1` construction, `2` maintenance, `3` utility, `4` unknown work-zone type |
| `VE_FORMS` | In-transport motor vehicles | Integer count |
| `PVH_INVL` | Parked or working vehicles involved | Integer count; `0` means none |

The readable `NAME` columns supplied by CRSS are not renamed or copied into the model table. The numeric code columns remain unchanged; `NAME` columns are reference documentation only.

## The cleaned accident table

This step selects only the needed columns and makes a copy. It does not filter rows or create `injury_involved`.

In [7]:
ACCIDENT_COLUMNS = [
    "CASENUM", "YEAR", "MAX_SEV",
    "WEIGHT", "PSU", "PSU_VAR", "PSUSTRAT",
    "MONTH", "DAY_WEEK", "HOUR", "LGT_COND", "WEATHER",
    "RELJCT2", "TYP_INT", "WRK_ZONE", "VE_FORMS", "PVH_INVL",
]

accidents_clean = accident[ACCIDENT_COLUMNS].copy()

print("Cleaned shape:", accidents_clean.shape)
print("Unique CASENUM values:", accidents_clean["CASENUM"].nunique())
print("Missing values:")
print(accidents_clean.isna().sum()[accidents_clean.isna().sum() > 0])
accidents_clean.head(10)

Cleaned shape: (51658, 17)
Unique CASENUM values: 51658
Missing values:
Series([], dtype: int64)


,CASENUM,YEAR,MAX_SEV,WEIGHT,PSU,PSU_VAR,PSUSTRAT,MONTH,DAY_WEEK,HOUR,LGT_COND,WEATHER,RELJCT2,TYP_INT,WRK_ZONE,VE_FORMS,PVH_INVL
0,202405532075,2024,0,224.394228,33,33,11,1,2,19,3,1,3,2,0,2,0
1,202405532153,2024,0,130.016411,22,22,5,1,3,15,1,1,3,3,0,2,0
2,202405532158,2024,0,211.951916,39,39,7,1,3,8,1,10,18,1,0,3,0
3,202405533474,2024,0,126.198089,25,25,8,1,2,14,1,10,2,3,0,2,0
4,202405533487,2024,0,218.212499,25,25,8,1,3,19,2,10,1,1,0,1,0
5,202405533497,2024,2,129.754166,47,47,9,1,2,15,1,10,2,2,0,2,0
6,202405533502,2024,1,125.932296,67,67,8,1,3,8,1,10,2,2,0,2,0
7,202405533521,2024,1,47.654018,83,83,13,1,2,19,3,1,2,3,0,2,0
8,202405533542,2024,2,130.542807,83,83,13,1,2,20,3,1,2,3,2,2,0
9,202405533564,2024,0,223.903742,83,83,13,1,2,8,1,1,1,1,0,1,0


## we validate the cleaned accident table

In [8]:
if accidents_clean["CASENUM"].duplicated().any():
    raise ValueError("CASENUM is duplicated")

print("CASENUM is unique for all", len(accidents_clean), "rows.")

CASENUM is unique for all 51658 rows.


In [9]:
if not accidents_clean["YEAR"].eq(2024).all():
    raise ValueError("Cleaned accident table contains a year other than 2024")

print("All rows belong to the 2024 source release.")

All rows belong to the 2024 source release.


In [10]:
survey_fields = ["WEIGHT", "PSU", "PSU_VAR", "PSUSTRAT"]

if accidents_clean[survey_fields].isna().any().any():
    raise ValueError("A sampling or survey-design field is missing")

if (accidents_clean["WEIGHT"] <= 0).any():
    raise ValueError("WEIGHT must be positive")

print("Survey fields are complete and all WEIGHT values are positive.")

Survey fields are complete and all WEIGHT values are positive.


## We define the target and eligible crash population

`MAX_SEV` is the source variable for the binary outcome.

| `MAX_SEV` | Meaning | Project treatment |
|---|---|---|
| `0` | No apparent injury | Target `0` |
| `1`-`5` | Reported injury of any severity | Target `1` |
| `6` | Died prior to crash | Exclude |
| `8` | No person involved | Exclude |
| `9` | Unknown or not reported | Exclude |

The resulting target means **at least one reported injury**, not medically verified injury or crash severity.

In [11]:
expected_max_sev = {0, 1, 2, 3, 4, 5, 6, 8, 9}
unexpected = set(accidents_clean["MAX_SEV"].dropna().unique()) - expected_max_sev

if unexpected:
    raise ValueError(f"Unexpected MAX_SEV code(s): {unexpected}")

print(accidents_clean["MAX_SEV"].value_counts().sort_index())

MAX_SEV
0    24732
1    10139
2     9198
3     5489
4     1096
5      127
6        3
8       17
9      857
Name: count, dtype: int64


In [12]:
eligible = accidents_clean["MAX_SEV"].isin([0, 1, 2, 3, 4, 5])

eligible_accidents = accidents_clean.loc[eligible].copy()
eligible_accidents["injury_involved"] = (
    eligible_accidents["MAX_SEV"].isin([1, 2, 3, 4, 5]).astype("int8")
)

print("Eligible crashes:", len(eligible_accidents))
eligible_accidents.head(10)

Eligible crashes: 50781


,CASENUM,YEAR,MAX_SEV,WEIGHT,PSU,PSU_VAR,PSUSTRAT,MONTH,DAY_WEEK,HOUR,LGT_COND,WEATHER,RELJCT2,TYP_INT,WRK_ZONE,VE_FORMS,PVH_INVL,injury_involved
0,202405532075,2024,0,224.394228,33,33,11,1,2,19,3,1,3,2,0,2,0,0
1,202405532153,2024,0,130.016411,22,22,5,1,3,15,1,1,3,3,0,2,0,0
2,202405532158,2024,0,211.951916,39,39,7,1,3,8,1,10,18,1,0,3,0,0
3,202405533474,2024,0,126.198089,25,25,8,1,2,14,1,10,2,3,0,2,0,0
4,202405533487,2024,0,218.212499,25,25,8,1,3,19,2,10,1,1,0,1,0,0
5,202405533497,2024,2,129.754166,47,47,9,1,2,15,1,10,2,2,0,2,0,1
6,202405533502,2024,1,125.932296,67,67,8,1,3,8,1,10,2,2,0,2,0,1
7,202405533521,2024,1,47.654018,83,83,13,1,2,19,3,1,2,3,0,2,0,1
8,202405533542,2024,2,130.542807,83,83,13,1,2,20,3,1,2,3,2,2,0,1
9,202405533564,2024,0,223.903742,83,83,13,1,2,8,1,1,1,1,0,1,0,0


### Compare target prevalence with and without survey weights

The row count answers what appears in the 2024 CRSS sample. The survey-weighted distribution estimates the represented eligible population more closely.

The weights do not change the target or become predictors.

In [13]:
target_summary = eligible_accidents.groupby("injury_involved").agg(
    crashes=("CASENUM", "size"),
    sum_of_weights=("WEIGHT", "sum"),
).reset_index()

total_crashes = target_summary["crashes"].sum()
total_weight = target_summary["sum_of_weights"].sum()

target_summary["row_share"] = target_summary["crashes"] / total_crashes
target_summary["weighted_share"] = target_summary["sum_of_weights"] / total_weight
target_summary["meaning"] = target_summary["injury_involved"].map({
    0: "No reported injury",
    1: "Reported injury",
})

target_summary[["injury_involved", "meaning", "crashes", "row_share", "sum_of_weights", "weighted_share"]]

,injury_involved,meaning,crashes,row_share,sum_of_weights,weighted_share
0,0,No reported injury,24732,0.487033,4.376542e+06,0.728873
1,1,Reported injury,26049,0.512967,1.627994e+06,0.271127


## Explore vehicle.csv

In [14]:
vehicle = raw["vehicle"]

print("Shape:", vehicle.shape)
print("Columns:")
print(vehicle.columns.tolist())

Shape: (90641, 167)
Columns:
['CASENUM', 'VEH_NO', 'PSU', 'PSU_VAR', 'PSUSTRAT', 'REGION', 'REGIONNAME', 'URBANICITY', 'URBANICITYNAME', 'STRATUM', 'STRATUMNAME', 'PJ', 'WEIGHT', 'VE_FORMS', 'MONTH', 'MONTHNAME', 'HOUR', 'HOURNAME', 'MINUTE', 'MINUTENAME', 'HARM_EV', 'HARM_EVNAME', 'MAN_COLL', 'MAN_COLLNAME', 'NUMOCCS', 'NUMOCCSNAME', 'UNITTYPE', 'UNITTYPENAME', 'HIT_RUN', 'HIT_RUNNAME', 'VIN', 'VINNAME', 'MOD_YEAR', 'MOD_YEARNAME', 'MDLYR_IM', 'MDLYR_IMNAME', 'VPICMAKE', 'VPICMAKENAME', 'VPICMODEL', 'VPICMODELNAME', 'VPICBODYCLASS', 'VPICBODYCLASSNAME', 'MAKE', 'MAKENAME', 'MODEL', 'BODY_TYP', 'BODY_TYPNAME', 'ICFINALBODY', 'ICFINALBODYNAME', 'GVWR_FROM', 'GVWR_FROMNAME', 'GVWR_TO', 'GVWR_TONAME', 'TOW_VEH', 'TOW_VEHNAME', 'TRLR1VIN', 'TRLR1VINNAME', 'TRLR2VIN', 'TRLR2VINNAME', 'TRLR3VIN', 'TRLR3VINNAME', 'TRLR1GVWR', 'TRLR1GVWRNAME', 'TRLR2GVWR', 'TRLR2GVWRNAME', 'TRLR3GVWR', 'TRLR3GVWRNAME', 'J_KNIFE', 'J_KNIFENAME', 'MCARR_ID', 'MCARR_IDNAME', 'MCARR_I1', 'MCARR_I1NAME', 'MCARR_I2'

### Vehicle-table first rows

In [15]:
vehicle.head(10)

,CASENUM,VEH_NO,PSU,PSU_VAR,PSUSTRAT,REGION,REGIONNAME,URBANICITY,URBANICITYNAME,STRATUM,...,P_CRASH2,P_CRASH2NAME,P_CRASH3,P_CRASH3NAME,PCRASH4,PCRASH4NAME,PCRASH5,PCRASH5NAME,ACC_TYPE,ACC_TYPENAME
0,202405532075,1,33,33,11,3,"South (MD, DE, DC, WV, VA, KY, TN, NC, SC, GA,...",1,Urban Area,10,...,53,Traveling in same direction with higher speed,1,No Avoidance Maneuver,1,Tracking,1,Stayed in original travel lane,21,"D21-Same Trafficway, Same Direction-Rear End-S..."
1,202405532075,2,33,33,11,3,"South (MD, DE, DC, WV, VA, KY, TN, NC, SC, GA,...",1,Urban Area,10,...,50,Other vehicle stopped,99,Unknown/Not Reported,1,Tracking,1,Stayed in original travel lane,20,"D20-Same Trafficway, Same Direction-Rear End-S..."
2,202405532153,1,22,22,5,1,"Northeast (PA, NJ, NY, NH, VT, RI, MA, ME, CT)",2,Rural Area,9,...,11,Over the lane line on right side of travel lane,99,Unknown/Not Reported,1,Tracking,2,"Stayed on roadway, but left original travel lane",46,"F46-Same Trafficway, Same Direction-Angle, Sid..."
3,202405532153,2,22,22,5,1,"Northeast (PA, NJ, NY, NH, VT, RI, MA, ME, CT)",2,Rural Area,9,...,60,From adjacent lane (same direction) over left ...,99,Unknown/Not Reported,1,Tracking,1,Stayed in original travel lane,45,"F45-Same Trafficway, Same Direction-Angle, Sid..."
4,202405532158,1,39,39,7,2,"Midwest (OH, IN, IL, MI, WI, MN, ND, SD, NE, I...",1,Urban Area,10,...,78,Encroachment by other vehicle - details unknown,99,Unknown/Not Reported,5,"Skidding Laterally, Rotation Direction Unknown",2,"Stayed on roadway, but left original travel lane",49,"F49-Same Trafficway, Same Direction-Angle, Sid..."
5,202405532158,2,39,39,7,2,"Midwest (OH, IN, IL, MI, WI, MN, ND, SD, NE, I...",1,Urban Area,10,...,19,Unknown travel direction,99,Unknown/Not Reported,1,Tracking,2,"Stayed on roadway, but left original travel lane",49,"F49-Same Trafficway, Same Direction-Angle, Sid..."
6,202405532158,3,39,39,7,2,"Midwest (OH, IN, IL, MI, WI, MN, ND, SD, NE, I...",1,Urban Area,10,...,78,Encroachment by other vehicle - details unknown,99,Unknown/Not Reported,1,Tracking,3,"Stayed on roadway, not known if left original ...",98,M98-Other Crash Type
7,202405533474,1,25,25,8,2,"Midwest (OH, IN, IL, MI, WI, MN, ND, SD, NE, I...",1,Urban Area,9,...,65,"From crossing street, turning into same direction",99,Unknown/Not Reported,1,Tracking,1,Stayed in original travel lane,77,K77-Trafficway Vehicle Turning-Turn Into Path-...
8,202405533474,2,25,25,8,2,"Midwest (OH, IN, IL, MI, WI, MN, ND, SD, NE, I...",1,Urban Area,9,...,15,Turning Left,99,Unknown/Not Reported,1,Tracking,1,Stayed in original travel lane,76,K76-Trafficway Vehicle Turning-Turn Into Path-...
9,202405533487,1,25,25,8,2,"Midwest (OH, IN, IL, MI, WI, MN, ND, SD, NE, I...",1,Urban Area,10,...,89,Animal -unknown location,99,Unknown/Not Reported,1,Tracking,1,Stayed in original travel lane,13,C13-Single Driver-Forward Impact-Pedestrian/ A...


### Vehicle-table missing values

In [16]:
vehicle.isna().sum().sort_values(ascending=False).to_frame("missing_values")

,missing_values
MCARR_I2NAME,1176
MCARR_I1NAME,1176
MCARR_IDNAME,1176
NUMOCCSNAME,174
PSU_VAR,0
...,...
PCRASH4NAME,0
PCRASH5,0
PCRASH5NAME,0
ACC_TYPE,0


### Vehicle identifier uniqueness

`CASENUM` identifies the crash, and `VEH_NO` identifies a vehicle within that crash. Together, `CASENUM + VEH_NO` should uniquely identify one vehicle record.

In [17]:
vehicle_keys = ["CASENUM", "VEH_NO"]

print("Vehicle rows:", len(vehicle))
print("Unique CASENUM values:", vehicle["CASENUM"].nunique())
print("Unique CASENUM + VEH_NO pairs:", vehicle[vehicle_keys].drop_duplicates().shape[0])

if vehicle[vehicle_keys].isna().any().any():
    raise ValueError("A vehicle identifier is missing")

if vehicle.duplicated(vehicle_keys).any():
    raise ValueError("A CASENUM + VEH_NO vehicle key is duplicated")

Vehicle rows: 90641
Unique CASENUM values: 51658
Unique CASENUM + VEH_NO pairs: 90641


## Selected vehicle fields and coding
From our project proposal we derive vehicle features from six fields. This section documents what each field represents before aggregation.

| Field | Meaning | Relevant project use |
|---|---|---|
| `BODY_TYP` | Vehicle body type | Identify motorcycles and large trucks |
| `TOW_VEH` | Towing status or trailing-unit information | Resolve uncertain body type `79` when determining large trucks |
| `MOD_YEAR` | Vehicle model year | Find the oldest known model year and identify unknown model years |
| `VSPD_LIM` | Posted speed limit | Find the highest valid limit and identify unknown or special speed settings |
| `VSURCOND` | Roadway surface condition | Identify wet/water or snow/ice/slush conditions |
| `VALIGN` | Roadway alignment | Identify curves |

### Unknown and special codes

These fields may contain values that are not ordinary measurements:

- `MOD_YEAR`: `9998` or `9999`, and missing values, mean the model year is unknown.
- `VSPD_LIM`: valid posted limits are `5`-`95` in steps of `5`; `0` means a special road setting, while `98`, `99`, or missing indicate unknown information.
- `VSURCOND`: unknown values are `98`, `99`, or missing; `0` may represent a non-roadway or not-applicable situation.
- `VALIGN`: `8` or `9`, and missing values, indicate unknown alignment; `0` may represent a non-roadway or not-applicable situation.
- `BODY_TYP`: `98` or `99`, and missing values, indicate unknown body type.

## Inspect the six selected vehicle fields individually

In [18]:
vehicle_fields = ["BODY_TYP", "TOW_VEH", "MOD_YEAR", "VSPD_LIM", "VSURCOND", "VALIGN"]

vehicle_field_profile = pd.DataFrame({
    "field": vehicle_fields,
    "unique_codes": [vehicle[field].nunique(dropna=False) for field in vehicle_fields],
    "missing_values": [vehicle[field].isna().sum() for field in vehicle_fields],
})

vehicle_field_profile

,field,unique_codes,missing_values
0,BODY_TYP,62,0
1,TOW_VEH,9,0
2,MOD_YEAR,75,0
3,VSPD_LIM,19,0
4,VSURCOND,13,0
5,VALIGN,7,0


## Aggregating vehicle records to one crash-level row

The final modeling unit remains one eligible crash. The vehicle table is converted into a separate `vehicle_summaries` table with one row per `CASENUM`, then checked before joining.

In [19]:
import numpy as np


def vehicle_presence(case_ids, positive, uncertain, inapplicable=None):
    level = pd.Series(1, index=case_ids.index)

    if inapplicable is not None:
        level.loc[inapplicable] = 0

    level.loc[uncertain] = 2
    level.loc[positive] = 3

    return (
        level.groupby(case_ids)
        .max()
        .map({
            0: "not_applicable",
            1: "no",
            2: "unknown",
            3: "yes",
        })
    )

### Numeric vehicle summaries

- `max_posted_mph` keeps the highest valid posted limit from `5` through `95`.
- `oldest_model_year` keeps the minimum valid model year.
- `speed_limit_unknown` preserves unknown or missing speed-limit evidence.
- `speed_limit_special` preserves code `0` as a special setting.
- `model_year_unknown` preserves unknown model-year evidence.

Unknown or special codes are not used as ordinary numeric measurements.

In [20]:
case = vehicle["CASENUM"]
speed = vehicle["VSPD_LIM"]
surface = vehicle["VSURCOND"]
alignment = vehicle["VALIGN"]
body = vehicle["BODY_TYP"]
towing = vehicle["TOW_VEH"]
year = vehicle["MOD_YEAR"]

valid_speed = speed.isin(range(5, 96, 5))
unknown_speed = speed.isna() | speed.isin([98, 99])
unknown_year = year.isna() | year.isin([9998, 9999])

body_unknown = body.isna() | body.isin([98, 99])
surface_unknown = surface.isna() | surface.isin([98, 99])
alignment_unknown = alignment.isna() | alignment.isin([8, 9])

truck = body.isin([60, 61, 62, 63, 64, 66, 67, 71, 72, 78])
truck |= body.eq(79) & towing.isin([1, 2, 3, 4])
truck_uncertain = body_unknown | (body.eq(79) & ~towing.isin([0, 1, 2, 3, 4, 5, 6, 9]))

vehicle_summaries = pd.DataFrame({
    "max_posted_mph": speed.where(valid_speed).groupby(case).max(),
    "speed_limit_unknown": unknown_speed.groupby(case).any(),
    "speed_limit_special": speed.eq(0).groupby(case).any(),
    "wet_or_water": vehicle_presence(case, surface.isin([2, 6]), surface_unknown, surface.eq(0)),
    "snow_ice_or_slush": vehicle_presence(case, surface.isin([3, 4, 10]), surface_unknown, surface.eq(0)),
    "curve": vehicle_presence(case, alignment.isin([2, 3, 4]), alignment_unknown, alignment.eq(0)),
    "motorcycle": vehicle_presence(case, body.between(80, 89), body_unknown),
    "large_truck": vehicle_presence(case, truck, truck_uncertain),
    "oldest_model_year": year.mask(unknown_year).groupby(case).min().astype("Int64"),
    "model_year_unknown": unknown_year.groupby(case).any(),
})

print("Vehicle summary rows:", len(vehicle_summaries))
print("Unique CASENUM values:", vehicle_summaries.index.nunique())
vehicle_summaries.head(10)

Vehicle summary rows: 51658
Unique CASENUM values: 51658


,max_posted_mph,speed_limit_unknown,speed_limit_special,wet_or_water,snow_ice_or_slush,curve,motorcycle,large_truck,oldest_model_year,model_year_unknown
CASENUM,,,,,,,,,,
202405532075,45.0,False,False,no,no,no,no,no,2001,False
202405532153,35.0,False,False,no,no,no,no,no,2014,False
202405532158,55.0,False,False,no,no,yes,no,no,2002,False
202405533474,25.0,False,False,yes,no,no,no,no,2018,False
202405533487,55.0,False,False,no,no,no,no,no,2017,False
202405533497,35.0,False,False,no,no,no,no,no,2002,False
202405533502,35.0,False,False,unknown,unknown,no,no,no,2010,False
202405533521,40.0,False,False,no,no,no,no,no,2013,False
202405533542,35.0,False,False,no,no,no,no,no,2010,False


## Next we explore person.csv

Now inspect the person table independently. First establish its size and source columns without defining person features or joining records.

In [21]:
person = raw["person"]

print("Shape:", person.shape)
print("Columns:")
print(person.columns.tolist())

Shape: (126159, 112)
Columns:
['CASENUM', 'VE_FORMS', 'VEH_NO', 'PER_NO', 'PSU', 'PSU_VAR', 'PSUSTRAT', 'REGION', 'REGIONNAME', 'URBANICITY', 'URBANICITYNAME', 'STRATUM', 'STRATUMNAME', 'PJ', 'WEIGHT', 'MONTH', 'MONTHNAME', 'HOUR', 'HOURNAME', 'MINUTE', 'MINUTENAME', 'HARM_EV', 'HARM_EVNAME', 'MAN_COLL', 'MAN_COLLNAME', 'SCH_BUS', 'SCH_BUSNAME', 'MOD_YEAR', 'MOD_YEARNAME', 'VPICMAKE', 'VPICMAKENAME', 'VPICMODEL', 'VPICMODELNAME', 'VPICBODYCLASS', 'VPICBODYCLASSNAME', 'MAKE', 'MAKENAME', 'BODY_TYP', 'BODY_TYPNAME', 'ICFINALBODY', 'ICFINALBODYNAME', 'GVWR_FROM', 'GVWR_FROMNAME', 'GVWR_TO', 'GVWR_TONAME', 'TOW_VEH', 'TOW_VEHNAME', 'AGE', 'AGENAME', 'AGE_IM', 'AGE_IMNAME', 'SEX', 'SEXNAME', 'SEX_IM', 'SEX_IMNAME', 'PER_TYP', 'PER_TYPNAME', 'DEVTYPE', 'DEVTYPENAME', 'DEVMOTOR', 'DEVMOTORNAME', 'INJ_SEV', 'INJ_SEVNAME', 'INJSEV_IM', 'INJSEV_IMNAME', 'SEAT_POS', 'SEAT_POSNAME', 'SEAT_IM', 'SEAT_IMNAME', 'REST_USE', 'REST_USENAME', 'REST_MIS', 'REST_MISNAME', 'HELM_USE', 'HELM_USENAME', 'HELM_

### Person-table first rows

In [22]:
person.head(10)

,CASENUM,VE_FORMS,VEH_NO,PER_NO,PSU,PSU_VAR,PSUSTRAT,REGION,REGIONNAME,URBANICITY,...,EMER_USE,EMER_USENAME,ROLLOVER,ROLLOVERNAME,IMPACT1,IMPACT1NAME,FIRE_EXP,FIRE_EXPNAME,MAK_MOD,MAK_MODNAME
0,202405532075,2,1,1,33,33,11,3,"South (MD, DE, DC, WV, VA, KY, TN, NC, SC, GA,...",1,...,0.0,Not Applicable,0.0,No Rollover,6.0,6 Clock Point,0.0,No or Not Reported,35047.0,Nissan/Datsun Altima
1,202405532075,2,2,1,33,33,11,3,"South (MD, DE, DC, WV, VA, KY, TN, NC, SC, GA,...",1,...,0.0,Not Applicable,0.0,No Rollover,12.0,12 Clock Point,0.0,No or Not Reported,52401.0,Mitsubishi Montero/Montero Sport
2,202405532153,2,1,1,22,22,5,1,"Northeast (PA, NJ, NY, NH, VT, RI, MA, ME, CT)",2,...,0.0,Not Applicable,0.0,No Rollover,1.0,1 Clock Point,0.0,No or Not Reported,55401.0,Hyundai Santa Fe
3,202405532153,2,2,1,22,22,5,1,"Northeast (PA, NJ, NY, NH, VT, RI, MA, ME, CT)",2,...,0.0,Not Applicable,0.0,No Rollover,9.0,9 Clock Point,0.0,No or Not Reported,7482.0,Dodge Ram Pickup
4,202405532158,3,1,1,39,39,7,2,"Midwest (OH, IN, IL, MI, WI, MN, ND, SD, NE, I...",1,...,0.0,Not Applicable,0.0,No Rollover,12.0,12 Clock Point,0.0,No or Not Reported,20401.0,Chevrolet S-10 Blazer/TrailBlazer (2002 only; ...
5,202405532158,3,2,1,39,39,7,2,"Midwest (OH, IN, IL, MI, WI, MN, ND, SD, NE, I...",1,...,0.0,Not Applicable,0.0,No Rollover,10.0,10 Clock Point,0.0,No or Not Reported,37441.0,Honda Odyssey
6,202405532158,3,3,1,39,39,7,2,"Midwest (OH, IN, IL, MI, WI, MN, ND, SD, NE, I...",1,...,0.0,Not Applicable,0.0,No Rollover,9.0,9 Clock Point,0.0,No or Not Reported,41051.0,Mazda Mazda3
7,202405533474,2,1,1,25,25,8,2,"Midwest (OH, IN, IL, MI, WI, MN, ND, SD, NE, I...",1,...,0.0,Not Applicable,0.0,No Rollover,7.0,7 Clock Point,0.0,No or Not Reported,12402.0,Ford Escape
8,202405533474,2,2,1,25,25,8,2,"Midwest (OH, IN, IL, MI, WI, MN, ND, SD, NE, I...",1,...,0.0,Not Applicable,0.0,No Rollover,1.0,1 Clock Point,0.0,No or Not Reported,20404.0,Chevrolet Equinox
9,202405533487,1,1,1,25,25,8,2,"Midwest (OH, IN, IL, MI, WI, MN, ND, SD, NE, I...",1,...,0.0,Not Applicable,0.0,No Rollover,12.0,12 Clock Point,0.0,No or Not Reported,7442.0,Dodge Caravan/Grand Caravan


### Person-table missing values

In [23]:
person.isna().sum().sort_values(ascending=False).to_frame("missing_values")

,missing_values
VPICMODELNAME,5249
VPICBODYCLASS,5249
VPICBODYCLASSNAME,5249
MAKE,5249
MAKENAME,5249
...,...
DRINKINGNAME,0
HOSPITALNAME,0
STR_VEH,0
LOCATION,0


### Person identifier uniqueness

One person is identified within a crash and vehicle by:

```text
CASENUM + VEH_NO + PER_NO
```

A person without an in-transport vehicle may still have `VEH_NO = 0`. This is important when interpreting the table, but it does not make the person key invalid.

In [24]:
person_keys = ["CASENUM", "VEH_NO", "PER_NO"]

print("Person rows:", len(person))
print("Unique person keys:", person[person_keys].drop_duplicates().shape[0])

if person[person_keys].isna().any().any():
    raise ValueError("A person identifier is missing")

if person.duplicated(person_keys).any():
    raise ValueError("A CASENUM + VEH_NO + PER_NO person key is duplicated")

print("Person key values are complete and unique.")

Person rows: 126159
Unique person keys: 126159
Person key values are complete and unique.


## Selected person fields and coding

The proposal derives person-level summaries from two source fields:

| Field | Meaning | Project use |
|---|---|---|
| `PER_TYP` | Person type or role | Identify pedestrians, pedalcyclists, drivers, and unknown occupant roles |
| `AGE` | Person age | Summarize the youngest and oldest known identified drivers |

### `PER_TYP` codes used by the project

| Code | Meaning | Project use |
|---|---|---|
| `1` | Driver of a motor vehicle in transport | Included in driver-age summaries |
| `5` | Pedestrian | Creates `pedestrian` presence |
| `6` | Bicyclist | Creates `pedalcyclist` presence |
| `7` | Other pedalcyclist | Creates `pedalcyclist` presence |
| `9` | Unknown occupant type | Creates `unknown_occupant_role` presence |
| `19` | Unknown type of nonmotorist | Makes pedestrian/pedalcyclist presence uncertain |
| Missing | Person type unknown | Makes relevant presence uncertain |

### `AGE` codes used by the project

- Valid ages are `0` through `120`.
- `0` means less than one year old.
- `998` means not known.
- `999` is unknown or not reported.
- Missing values indicate unavailable age evidence.

Only `PER_TYP == 1` contributes to driver-age summaries. Unknown ages are excluded from minimum and maximum calculations.

### Six resulting crash-level person features

| Feature | Meaning |
|---|---|
| `pedestrian` | Whether any pedestrian is confirmed, unresolved, absent, or not applicable |
| `pedalcyclist` | Whether any pedalcyclist is confirmed, unresolved, absent, or not applicable |
| `youngest_driver_age` | Minimum valid age among identified drivers |
| `oldest_driver_age` | Maximum valid age among identified drivers |
| `driver_age_status` | Whether identified driver ages are all known, partly known, all unknown, or absent |
| `unknown_occupant_role` | Whether a person has unknown occupant role code `9` |

## Aggregate person records to one crash-level row

The person table is aggregated by `CASENUM`, not joined directly to vehicle records. This preserves pedestrians and other people whose `VEH_NO` is `0` or is not an in-transport vehicle key.

In [25]:
def person_presence(case_ids, positive, uncertain):
    level = pd.Series(1, index=case_ids.index)

    level.loc[uncertain] = 2
    level.loc[positive] = 3

    return (
        level.groupby(case_ids)
        .max()
        .map({
            1: "no",
            2: "unknown",
            3: "yes",
        })
    )

### Driver-age rules

- Only `PER_TYP == 1` is a driver.
- Valid ages are `0` through `120`.
- Ages `998`, `999`, and missing values are excluded from numerical minimum and maximum calculations.
- `0` remains a valid age meaning less than one year old.

The driver-age status summarizes whether identified driver ages are complete or incomplete.

In [26]:
person_case = person["CASENUM"]
role = person["PER_TYP"]
age = person["AGE"]

driver = role.eq(1)
driver_age = age.where(driver & age.between(0, 120))

driver_count = driver.groupby(person_case).sum()
known_driver_age_count = driver_age.groupby(person_case).count()

driver_age_status = pd.Series("all_known", index=driver_count.index)
driver_age_status.loc[known_driver_age_count < driver_count] = "partly_known"
driver_age_status.loc[known_driver_age_count.eq(0)] = "all_unknown"
driver_age_status.loc[driver_count.eq(0)] = "no_identified_driver"

uncertain_nonmotorist = role.isna() | role.eq(19)

person_summaries = pd.DataFrame({
    "pedestrian": person_presence(person_case, role.eq(5), uncertain_nonmotorist),
    "pedalcyclist": person_presence(person_case, role.isin([6, 7]), uncertain_nonmotorist),
    "youngest_driver_age": driver_age.groupby(person_case).min().astype("Int64"),
    "oldest_driver_age": driver_age.groupby(person_case).max().astype("Int64"),
    "driver_age_status": driver_age_status,
    "unknown_occupant_role": person_presence(person_case, role.eq(9), role.isna()),
})

print("Person summary rows:", len(person_summaries))
print("Unique CASENUM values:", person_summaries.index.nunique())
person_summaries.head(10)

Person summary rows: 51641
Unique CASENUM values: 51641


,pedestrian,pedalcyclist,youngest_driver_age,oldest_driver_age,driver_age_status,unknown_occupant_role
CASENUM,,,,,,
202405532075,no,no,33,47,all_known,no
202405532153,no,no,61,77,all_known,no
202405532158,no,no,28,77,all_known,no
202405533474,no,no,77,82,all_known,no
202405533487,no,no,34,34,all_known,no
202405533497,no,no,17,63,all_known,no
202405533502,no,no,25,32,all_known,no
202405533521,no,no,27,57,all_known,no
202405533542,no,no,23,55,all_known,no


## Join vehicle and person summaries to the eligible crashes

Vehicle and person records are aggregated independently before joining. This avoids multiplying person rows by vehicle rows and preserves people who do not have an in-transport vehicle key.

The left table is `eligible_accidents`, so the final table remains one row per eligible crash.

In [27]:
crash_table = eligible_accidents.join(
    vehicle_summaries,
    on="CASENUM",
    how="left",
    validate="one_to_one",
)

crash_table = crash_table.join(
    person_summaries,
    on="CASENUM",
    how="left",
    validate="one_to_one",
)

print("Eligible crash rows:", len(eligible_accidents))
print("Final combined rows:", len(crash_table))
print("Final combined columns:", len(crash_table.columns))

if len(crash_table) != len(eligible_accidents):
    raise ValueError("Crash-level joins changed the eligible crash population")

if not crash_table["CASENUM"].is_unique:
    raise ValueError("Final crash table contains duplicate CASENUM values")

crash_table.head(10)

Eligible crash rows: 50781
Final combined rows: 50781
Final combined columns: 34


,CASENUM,YEAR,MAX_SEV,WEIGHT,PSU,PSU_VAR,PSUSTRAT,MONTH,DAY_WEEK,HOUR,...,motorcycle,large_truck,oldest_model_year,model_year_unknown,pedestrian,pedalcyclist,youngest_driver_age,oldest_driver_age,driver_age_status,unknown_occupant_role
0,202405532075,2024,0,224.394228,33,33,11,1,2,19,...,no,no,2001,False,no,no,33,47,all_known,no
1,202405532153,2024,0,130.016411,22,22,5,1,3,15,...,no,no,2014,False,no,no,61,77,all_known,no
2,202405532158,2024,0,211.951916,39,39,7,1,3,8,...,no,no,2002,False,no,no,28,77,all_known,no
3,202405533474,2024,0,126.198089,25,25,8,1,2,14,...,no,no,2018,False,no,no,77,82,all_known,no
4,202405533487,2024,0,218.212499,25,25,8,1,3,19,...,no,no,2017,False,no,no,34,34,all_known,no
5,202405533497,2024,2,129.754166,47,47,9,1,2,15,...,no,no,2002,False,no,no,17,63,all_known,no
6,202405533502,2024,1,125.932296,67,67,8,1,3,8,...,no,no,2010,False,no,no,25,32,all_known,no
7,202405533521,2024,1,47.654018,83,83,13,1,2,19,...,no,no,2013,False,no,no,27,57,all_known,no
8,202405533542,2024,2,130.542807,83,83,13,1,2,20,...,no,no,2010,False,no,no,23,55,all_known,no
9,202405533564,2024,0,223.903742,83,83,13,1,2,8,...,no,no,2012,False,no,no,26,26,all_known,no


In [28]:
matched_vehicle_crashes = crash_table[vehicle_summaries.columns].notna().any(axis=1).sum()
matched_person_crashes = crash_table[person_summaries.columns].notna().any(axis=1).sum()

print("Crash rows matched to vehicle summaries:", matched_vehicle_crashes)
print("Crash rows matched to person summaries:", matched_person_crashes)

vehicle_feature_names = vehicle_summaries.columns.tolist()
person_feature_names = person_summaries.columns.tolist()

missing_vehicle_columns = [
    column for column in vehicle_feature_names
    if column not in crash_table.columns
]
missing_person_columns = [
    column for column in person_feature_names
    if column not in crash_table.columns
]

if missing_vehicle_columns:
    raise ValueError(f"Missing vehicle columns: {missing_vehicle_columns}")

if missing_person_columns:
    raise ValueError(f"Missing person columns: {missing_person_columns}")

print("Vehicle feature count:", len(vehicle_feature_names))
print("Person feature count:", len(person_feature_names))
print("Combined feature count:", len(vehicle_feature_names) + len(person_feature_names))

Crash rows matched to vehicle summaries: 50781
Crash rows matched to person summaries: 50781
Vehicle feature count: 10
Person feature count: 6
Combined feature count: 16


## Save the cleaned crash table
We  will load this CSV instead of repeating raw-table loading or feature engineering.

In [29]:
cleaned_crashes_path = Path("data/cleaned_crashes.csv")
crash_table.to_csv(cleaned_crashes_path, index=False)

print("Saved:", cleaned_crashes_path)
print("Rows:", len(crash_table))

Saved: data\cleaned_crashes.csv
Rows: 50781
